In [2]:
# Importante activar un entorno de ejecución con GPU (GPU T4)

# instalamos dependencias
!pip install transformers accelerate datasets -q
!pip install huggingface_hub -q

In [3]:
# Imports
import re
import time
import zipfile
import torch
import os

from datetime import datetime
from pathlib import Path
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from huggingface_hub import login
from google.colab import drive, files
from dotenv import load_dotenv

# Configuración de parámetros para la generación
MAX_NEW_TOKENS = 350
TEMPERATURE = 0.15
TOP_P = 0.9
REPETITION_PENALTY = 1.1

In [ ]:
# cargamos el token de huggingface, en mi caso esta en la carpeta MyDrive
drive.mount('/content/drive')
load_dotenv('/content/drive/MyDrive/.env')
token = os.getenv("HF_TOKEN")
login(token=token)

# cargamos el modelo, en nuestro caso Gemma2, esto puede tardar un par de minutos
model_id = "google/gemma-2-2b-it"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)
generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

### Self-evaluating / Auto-evaluación

En este bloque vamos a utilizar el propio modelo para que revise las respuestas generadas y las califique para su posterior refinamiento

In [5]:
# Funciones auxiliares

# Cargamos el corpus (Q línea 1, A línea 2, una linea en blanco) importante mantener este formato
def load_qa_from_txt(path):
    res = []
    with open(path, encoding="utf-8") as f:
        text = f.read()
    # separamos por bloques
    blocks = text.split("\n\n")
    for block in blocks:
        # si el bloque esta vacio pasamos
        if not block.strip():
            continue
        parts = block.split("\n", 1)
        # si no hay pregunta y respuesta pasamos al siguiente/ no tiene el formato
        if len(parts) != 2:
            continue
        # guardo por separdo la pregunta y la respuesta
        question, answer = parts
        # si esta completo la añadimos
        if question and answer:
            res.append({"question": question,"answer": answer})

    return res


# función auxiliar para guardar el corpus en el mismo formato
def save_questions_txt(qa_pairs, file_path):
    with open(file_path, "w", encoding="utf-8") as f:
        for pair in qa_pairs:
            question = " ".join(pair["question"].split())
            answer = " ".join(pair["answer"].split())
            f.write(f"{question}\n")
            f.write(f"{answer}\n\n")


In [ ]:
# Prompt de evaluación y clasificación de la repsuesta por nota
def score_prompt(question, answer):
    return f"""
    You are an expert evaluator in computer vision.

Evaluate the quality of the following answer to a technical question and assign a score from 1 to 5 according to this rubric:

5 = Clear, technically correct, well-structured, and didactic answer.
4 = Correct and clear answer, with a good explanation.
3 = Partially correct or incomplete answer but an acceptable answer.
2 = Superficial answer,containing conceptual errors or if the answer does not answer the question.
1 = Incorrect, irrelevant, or confusing answer.

Question:
{question}

Answer:
{answer}

Return EXACTLY two lines using the following format:
Score: X
Reason: <brief explanation of why this score was assigned>
""".strip()


# función para las obtener las notas
def score_qa(generator, question, answer):
    prompt = score_prompt(question, answer)
    output = generator(prompt)[0]["generated_text"]

    score_match = re.search(r"Score\s*[:\-]?\s*([1-5])", output)
    # Buscamos una nota valida, si no devolvemos 0
    if score_match:
        score = int(score_match.group(1))
    else:
      score = 0

    # Buscamos la o devolvemos un stirng
    reason_match = re.search(r"Reason\s*[:\-]?\s*(.*)",output)
    if reason_match:
        reason = reason_match.group(1).strip()
    else:
      reason = "No reason provided."

    return score, reason

# Función auxiliar para recopilar las notas y mostrar una distribución de las respuestas
def sum_scores(qa_list, score_counts, accepted_list=None, rejected_list=None):
    total = len(qa_list)

    # comprobamos si lo pasan o devolvemos none
    if accepted_list is not None:
        accepted = len(accepted_list)
    else:
      accepted = None

    # comprobamos si lo pasan o devolvemos none
    if rejected_list is not None:
        rejected = len(rejected_list)
    else:
      rejected = None

    # Mostramos por pantalla los valores
    print("\n===RESUMEN FINAL===")
    print(f"QA analizados: {total}")
    # calculamos la media de aceptados y rechazados
    if accepted is not None:
        print(f"QA aceptados: {accepted}/{total} ({(accepted/total*100 if total else 0):.1f} %)")
    if rejected is not None:
        print(f"QA rechazados: {rejected}/{total} ({(rejected/total*100 if total else 0):.1f} %)")

    # devolvemos la distribución
    print("\n===Distribución de scores===")
    for s in sorted(score_counts.keys()):
        c = score_counts[s]
        print(f"Score {s}: {c}/{total} ({(c/total*100 if total else 0):.2f}%)")


# poner download a True si queremos descargar los txt de las rechazadas para verlas, el umbral determina la calidad para "aprobar" y ser apta
def evaluation(generator, input_path,out_ok="qa_computervision_filtrados_aprobados.txt",out_bad="qa_computervision_rechazado.txt",threshold=3,download=False):
    # definimos la lista de preguntas, dos listas para filtrar y el recuento de puntuaciones
    qa_list = load_qa_from_txt(input_path)
    filtered_qa, rejected_qa = [], []
    score_counts = {i: 0 for i in range(1, 6)}

    for p in qa_list:
        # extraemos el par qa
        question = p["question"]
        answer = p["answer"]

        # evlausmos con el modelo y guardamos la puntuacion y reason
        s, reason = score_qa(generator, p["question"], p["answer"])
        # vamos contando
        score_counts[s] += 1

        # comprobamos si pasa el umbral de nota se guarda
        if s >= threshold:
          filtered_qa.append(p)
        else:
          rejected_qa.append({**p, "score": s, "reason": reason})
        #
           #(filtered_qa if s >= threshold else rejected_qa).append(
           # p if s >= threshold else {**p, "score": s, "reason": reason}
        #)


    # llammamos a la funcion de los scores para evaluar
    sum_scores(qa_list, score_counts, filtered_qa, rejected_qa)
    # guardamos los resultados
    save_questions_txt(filtered_qa, out_ok)
    save_questions_txt(rejected_qa, out_bad)
    # si queremos descargar estos archivos ponemos a TRUE el parametro download, lo he dejado a false por que al final en el zip se ve claro, pero dpy la opción por si acaso se quiere ver algo
    if download:
      files.download(out_ok); files.download(out_bad)

# ejecutamos el bloque
input_file = next(iter(files.upload()))
evaluation(generator, input_file)


In [7]:
with open("qa_computervision_rechazado.txt", "r", encoding="utf-8") as f:
    lines = f.readlines()
with open("qa_computervision_rechazado_preguntas.txt", "w", encoding="utf-8") as f:
    for i in range(0, len(lines), 3):
        if lines[i].strip():
            f.write(lines[i])

Aqui llamamos al mismo promt y metodo de generate con las rechazadas, como es exactamente el casi el mismo código que la generación de preguntas y respuestas lo he comprimido para que sea más legible este archivo, empleamos aquí para las refinadas un prompt



In [ ]:
drive.mount('/content/drive')
CONCEPTS_FILE = Path("/content/qa_computervision_rechazado_preguntas.txt")


# funcion auxiliar para leer las preguntas de un txt
def load_questions_from_txt(file_path):
    path = Path(file_path)

    if not path.exists():
        raise FileNotFoundError(f"No se encontró el txt: {file_path}")

    questions = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#"):
                questions.append(line)
    return questions


def build_prompt_theory(question):
    prompt = f"""
You are a university-level computer vision teaching assistant.

This is a NEW and INDEPENDENT question.
Do NOT repeat or reuse previous answers.

QUESTION:
{question}

When answering:
- explain the purpose of the concept,
- describe how it works at a conceptual level,
- mention typical use cases,
- discuss important limitations or common pitfalls.

Guidelines:
- do NOT include implementation code,
- do NOT invent OpenCV or NumPy function names,
- use precise but accessible language,
- avoid markdown headers or section titles,
- aim for 2 concise paragraphs (6–8 sentences total).

IMPORTANT:
Quantization refers to image intensity quantization, not model compression.
Answer strictly in the context of digital image representation.

Answer below. Do not repeat the question or instructions.

=== ANSWER START ===
""".strip()
    return prompt


def clean_text(text):
    if "=== ANSWER START ===" in text:
        text = text.split("=== ANSWER START ===", 1)[-1]

    if "=== ANSWER END ===" in text:
        text = text.split("=== ANSWER END ===", 1)[0]

    if "model" in text:
        text = text.split("model", 1)[-1]

    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"\*\*", "", text)
    return text.strip()


def normalize(text):
    text = text.replace("\n\n", "\n")
    text = text.replace("\n", "\\n")
    return text.strip()

def generate_answer(prompt):
    messages = [{"role": "user", "content": prompt}]

    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True,return_tensors="pt").to(model.device)
    output_ids = model.generate(**inputs,max_new_tokens=MAX_NEW_TOKENS,do_sample=True,temperature=TEMPERATURE,top_p=TOP_P,repetition_penalty=REPETITION_PENALTY,eos_token_id=tokenizer.eos_token_id,)
    raw_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    return clean_text(raw_text)



start_time = time.time()

questions = load_questions_from_txt(CONCEPTS_FILE)
qa_results = []

for question in tqdm(questions,desc="Generating QA Pairs",unit="q"):
    prompt = build_prompt_theory(question)
    answer = generate_answer(prompt)

    qa_results.append({"question": question,"answer": answer})

    time.sleep(0.3)


# guardamos las respuestas generadas refinadas
filename = "qa_computervision_rejected_refined.txt"

with open(filename, "w", encoding="utf-8") as f:
    for qa in qa_results:
          q = normalize(qa["question"])
          a = normalize(qa["answer"])
          f.write(q + "\n")
          f.write(a + "\n\n")

print(f"\n Archivo guardado en la ruta: {filename}")
print(f"Se han procesado {len(qa_results)} preguntas")



In [ ]:
# Reevaluamos una vez más las preguntas refinadas para ver si pasan el mínimo de calidad y se añaden al corpus si es el caso
evaluation(generator,"qa_computervision_rejected_refined.txt",out_ok="qa_computervision_refined_and_revaluated.txt",out_bad="qa_computervision_still_rejected.txt",
    threshold=3,download = False
)

In [ ]:
# =========================
# Exportamos las respuestas validas para el entrenamiento como un zip
# =========================
zip_name = "corpus_usable.zip"

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    z.write("qa_computervision_filtrados_aprobados.txt" )
    z.write("qa_computervision_refined_and_revaluated.txt")

files.download(zip_name)
print(f"Las repsuestas consideradas válidas para el corpus han sido guardadas en: {zip_name}")
